In [28]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [29]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [30]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [31]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [32]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [33]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [34]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [35]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 119,
 'tn': 2592,
 'fp': 45,
 'fn': 244,
 'misclassification_rate': 0.09633333333333334,
 'false_positive_rate': 0.017064846416382253,
 'false_negative_rate': 0.6721763085399449}

### Check results on the test set (new data not yet seen by the model)

In [36]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 36,
 'tn': 859,
 'fp': 15,
 'fn': 90,
 'misclassification_rate': 0.105,
 'false_positive_rate': 0.017162471395881007,
 'false_negative_rate': 0.7142857142857143}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

My model had a test misclassification rate of 10.5%, so overall it was able to classify most accounts correctly. That gives me a fair amount of confidence in the model, but I would still be careful relying on it completely because it still missed a number of bots. I think the model performs well overall, but there is still room for improvement, especially when it comes to correctly identifying actual bots.

### What are potential ramifications of false positives from the model?

A false positive would mean that a real person is classified as a bot. This could be a problem because a legitimate user might get flagged, restricted, or removed even though they did nothing wrong. If this happened too often, users could also lose trust in the system.

### What are potential ramifications of false negatives from the model?

A false negative would mean that a bot is classified as a real person. This could allow bots to stay active and continue posting spam, misleading information, or other automated content. If the model misses too many bots, it would make the system less effective at actually detecting them.